In [0]:

bronze = "abfss://bronze@databricktraveljournal.dfs.core.windows.net"
table = "activity_code"
bronze_table_parquet_path = f"{bronze}/{table}"

activity_code_df = spark.read.format("parquet")\
    .load(f"{bronze_table_parquet_path}")

display(activity_code_df)

activity_code,activity_name,created_at,emoji,flag,id,date_type,year,month,day,_rescued_data
15,rafting,null,🛶,false,15,2026-06-06,2026,6,6,"{""created_at"":""2026-06-06T01:12:00.513Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/activity_code/year=2026/month=6/day=6/part-00001-tid-3256170012956726398-1d61b334-ef41-43a8-a41e-56f318c527c5-14-1.c000.snappy.parquet""}"
16,trekking,null,🥾,false,16,2026-06-06,2026,6,6,"{""created_at"":""2026-06-06T01:12:00.513Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/activity_code/year=2026/month=6/day=6/part-00001-tid-3256170012956726398-1d61b334-ef41-43a8-a41e-56f318c527c5-14-1.c000.snappy.parquet""}"
17,snowboarding,null,🏂,false,17,2026-06-06,2026,6,6,"{""created_at"":""2026-06-06T01:12:00.513Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/activity_code/year=2026/month=6/day=6/part-00001-tid-3256170012956726398-1d61b334-ef41-43a8-a41e-56f318c527c5-14-1.c000.snappy.parquet""}"
18,horseback_riding,null,🐎,false,18,2026-06-06,2026,6,6,"{""created_at"":""2026-06-06T01:12:00.513Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/activity_code/year=2026/month=6/day=6/part-00001-tid-3256170012956726398-1d61b334-ef41-43a8-a41e-56f318c527c5-14-1.c000.snappy.parquet""}"
19,caving,null,🕳️,false,19,2026-06-06,2026,6,6,"{""created_at"":""2026-06-06T01:12:00.513Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/activity_code/year=2026/month=6/day=6/part-00001-tid-3256170012956726398-1d61b334-ef41-43a8-a41e-56f318c527c5-14-1.c000.snappy.parquet""}"
20,birdwatching,null,🦅,false,20,2026-06-06,2026,6,6,"{""created_at"":""2026-06-06T01:12:00.513Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/activity_code/year=2026/month=6/day=6/part-00001-tid-3256170012956726398-1d61b334-ef41-43a8-a41e-56f318c527c5-14-1.c000.snappy.parquet""}"
21,windsurfing,null,🏄,false,21,2026-06-06,2026,6,6,"{""created_at"":""2026-06-06T01:12:00.513Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/activity_code/year=2026/month=6/day=6/part-00001-tid-3256170012956726398-1d61b334-ef41-43a8-a41e-56f318c527c5-14-1.c000.snappy.parquet""}"
22,canyoning,null,🏞️,false,22,2026-06-06,2026,6,6,"{""created_at"":""2026-06-06T01:12:00.513Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/activity_code/year=2026/month=6/day=6/part-00001-tid-3256170012956726398-1d61b334-ef41-43a8-a41e-56f318c527c5-14-1.c000.snappy.parquet""}"
23,ziplining,null,🛤️,false,23,2026-06-06,2026,6,6,"{""created_at"":""2026-06-06T01:12:00.513Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/activity_code/year=2026/month=6/day=6/part-00001-tid-3256170012956726398-1d61b334-ef41-43a8-a41e-56f318c527c5-14-1.c000.snappy.parquet""}"
24,wildlife_safari,null,🦁,false,24,2026-06-06,2026,6,6,"{""created_at"":""2026-06-06T01:12:00.513Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/activity_code/year=2026/month=6/day=6/part-00001-tid-3256170012956726398-1d61b334-ef41-43a8-a41e-56f318c527c5-14-1.c000.snappy.parquet""}"


### Quality 

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, coalesce
from pyspark.sql.functions import col, coalesce, get_json_object, to_timestamp, lit, when


def transform_to_silver(bronze_df: DataFrame) -> DataFrame:
    df = bronze_df

            # date_type is a real date; created_at is "-" so we skip it
    df = df.withColumn("created_at",
                               when(col("created_at").isNull(),
                                    get_json_object(col("_rescued_data"),"$.created_at"))
                                    .otherwise(col("created_at"))
                                    )
    
    df = (
        df
        # if the year is 2024, replace it with 2026 (keeps month/day/time exactly)
        .withColumn(
            "created_at",
            F.when(
                F.col("created_at").startswith("2024"),
                F.regexp_replace("created_at", r"^2024", "2026")
            ).otherwise(F.col("created_at"))
        )
        .withColumn("created_at", F.to_timestamp("created_at"))
        .withColumn("date_type", F.to_date("created_at"))
    )

    # date_type is a real date; created_at is "-" so we skip it
    df = (
        df.withColumn("created_at", F.to_timestamp("created_at"))
          .withColumn("date_type", F.to_date("date_type"))
    )

    df = df.withColumn("flag", F.lower(F.trim(F.col("flag"))) == F.lit("true"))

    df = (
        df.withColumn("year",          F.expr("try_cast(year as int)"))
          .withColumn("month",         F.expr("try_cast(month as int)"))
          .withColumn("day",           F.expr("try_cast(day as int)"))
          .withColumn("id",            F.expr("try_cast(id as int)"))
          .withColumn("activity_code", F.expr("try_cast(activity_code as int)"))
        .withColumn("year",  F.when(F.col("year") == 2024, F.lit(2026)).otherwise(F.col("year")))

    )

    df = (
        df.withColumn("activity_name", F.trim(F.col("activity_name")))
          .withColumn("emoji",         F.trim(F.col("emoji")))
    )

    
    quality_df = df \
            .withColumn("_is_valid", coalesce(
                (col("activity_code").isNotNull()) &
                (col("activity_name").isNotNull()) &
                (col("created_at").isNotNull()) &
                (col("id").isNotNull()) &
                lit(True)
            ))

        # Separate valid and invalid records
    valid_df = quality_df.filter(col("_is_valid"))
    invalid_df = quality_df.filter(~col("_is_valid"))

        # Log invalid records for investigation
    if invalid_df.count() > 0:

            print(f"Quarantined {invalid_df.count()} invalid records")
    
    
    return valid_df.drop("_rescued_data","_is_valid")

df = transform_to_silver(activity_code_df)


In [0]:
df.printSchema()

root
 |-- activity_code: integer (nullable = true)
 |-- activity_name: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- emoji: string (nullable = true)
 |-- flag: boolean (nullable = true)
 |-- id: integer (nullable = true)
 |-- date_type: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)



In [0]:
df.display()

activity_code,activity_name,created_at,emoji,flag,id,date_type,year,month,day
15,rafting,2026-06-06T01:12:00.513Z,🛶,false,15,2026-06-06,2026,6,6
16,trekking,2026-06-06T01:12:00.513Z,🥾,false,16,2026-06-06,2026,6,6
17,snowboarding,2026-06-06T01:12:00.513Z,🏂,false,17,2026-06-06,2026,6,6
18,horseback_riding,2026-06-06T01:12:00.513Z,🐎,false,18,2026-06-06,2026,6,6
19,caving,2026-06-06T01:12:00.513Z,🕳️,false,19,2026-06-06,2026,6,6
20,birdwatching,2026-06-06T01:12:00.513Z,🦅,false,20,2026-06-06,2026,6,6
21,windsurfing,2026-06-06T01:12:00.513Z,🏄,false,21,2026-06-06,2026,6,6
22,canyoning,2026-06-06T01:12:00.513Z,🏞️,false,22,2026-06-06,2026,6,6
23,ziplining,2026-06-06T01:12:00.513Z,🛤️,false,23,2026-06-06,2026,6,6
24,wildlife_safari,2026-06-06T01:12:00.513Z,🦁,false,24,2026-06-06,2026,6,6


### Deduplicate

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def deduplicate_by_key(df, keycolumns, order_column, ascending=False):
    """
    
    Deduplicate a Dataframe by composite key, keeping the row with the highest (or lowest) value in order_column

    Args:
        df: Input DataFrame with duplicates
        key_columns: List of columns forming the composite key
        order_column: Column to break ties (e.g., updated_at)
        ascending: If True, keep the smallest order_column value
    """
    
    order_expr = (
        F.col(order_column).asc() if ascending else F.col(order_column).desc()
    )

    window_spec = Window.partitionBy(*keycolumns).orderBy(order_expr)

    return df.withColumn("rank", F.row_number().over(window_spec)).filter(
        F.col("rank") == 1
    ).drop("rank")

activity_code_df = deduplicate_by_key(df, ["id"], "created_at", ascending=True)



In [0]:
activity_code_df.display()

activity_code,activity_name,created_at,emoji,flag,id,date_type,year,month,day
1,hiking,2026-06-06T01:12:00.513Z,⛰️,false,1,2026-06-06,2026,6,6
2,kayaking,2026-06-06T01:12:00.513Z,🚣,false,2,2026-06-06,2026,6,6
3,biking,2026-06-06T01:12:00.513Z,🚴,false,3,2026-06-06,2026,6,6
4,swimming,2026-06-06T01:12:00.513Z,🏊,false,4,2026-06-06,2026,6,6
5,camping,2026-06-06T01:12:00.513Z,⛺,false,5,2026-06-06,2026,6,6
6,photography,2026-06-06T01:12:00.513Z,📷,false,6,2026-06-06,2026,6,6
7,diving,2026-06-06T01:12:00.513Z,🤿,false,7,2026-06-06,2026,6,6
8,surfing,2026-06-06T01:12:00.513Z,🏄,false,8,2026-06-06,2026,6,6
9,skiing,2026-06-06T01:12:00.513Z,⛷️,false,9,2026-06-06,2026,6,6
10,climbing,2026-06-06T01:12:00.513Z,🧗,false,10,2026-06-06,2026,6,6


## Data Writing

In [0]:
activity_code_df.write.format("delta").mode("overwrite").save("abfss://silver@databricktraveljournal.dfs.core.windows.net/activity_code")

/databricks/python/lib/python3.12/site-packages/IPython/core/completer.py:2913: ProvisionalCompleterWarning: ``Completion`` is a provisional API (as of IPython 6.0). It may change without warnings. Use in corresponding context manager.
  yield Completion(start=offset - delta,
/databricks/python/lib/python3.12/site-packages/IPython/core/completer.py:2913: ProvisionalCompleterWarning: ``Completion`` is a provisional API (as of IPython 6.0). It may change without warnings. Use in corresponding context manager.
  yield Completion(start=offset - delta,
/databricks/python/lib/python3.12/site-packages/IPython/core/completer.py:2913: ProvisionalCompleterWarning: ``Completion`` is a provisional API (as of IPython 6.0). It may change without warnings. Use in corresponding context manager.
  yield Completion(start=offset - delta,
/databricks/python/lib/python3.12/site-packages/IPython/core/completer.py:2913: ProvisionalCompleterWarning: ``Completion`` is a provisional API (as of IPython 6.0). It 

## Delta

In [0]:
%sql

CREATE TABLE IF NOT EXISTS travel_journal_catalog.silver.activity_code
USING DELTA
LOCATION "abfss://silver@databricktraveljournal.dfs.core.windows.net/activity_code"